# The Sound of Data
## Analisi del dataset Spotify

Questo notebook verrà sviluppato passo passo. Prima di scegliere l'algoritmo, verificare sulle slide del corso quale metodo è richiesto come modello della percezione umana.


## 1. Obiettivo del progetto

A partire da un dataset di brani musicali estratti da Spotify, ciascuno corredato da caratteristiche sonore e musicali e associato a uno specifico genere, il progetto si propone di analizzare il modo in cui i brani possono essere organizzati e raggruppati sulla base delle loro proprietà audio.

In particolare, l'obiettivo è indagare se la suddivisione dei brani in generi musicali rispecchi una struttura riconoscibile, basata su caratteristiche musicali e acustiche misurabili, oppure se sia il risultato di un processo di categorizzazione cognitiva e percettiva operato dalla mente umana.

Attraverso l'applicazione di tecniche di clustering, si intende individuare gruppi di brani caratterizzati da proprietà sonore simili, considerando variabili come danceability, energy, valence, acousticness e tempo, senza utilizzare direttamente il genere musicale nella fase di raggruppamento.

I risultati ottenuti verranno successivamente analizzati e confrontati con i generi musicali originali, al fine di individuare eventuali corrispondenze o discrepanze tra le caratteristiche oggettive dei brani e la loro classificazione.

Il progetto mira quindi a comprendere come le caratteristiche sonore possano contribuire all'organizzazione della musica in categorie e quale relazione esista tra la struttura dei dati e i processi di percezione e categorizzazione umana.


In [ ]:
!pip install -q kagglehub

In [ ]:
# Librerie principali
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


## 2. Caricamento del dataset

In questa fase carichiamo il dataset Spotify Tracks Dataset, disponibile su Kaggle.

Utilizzeremo la libreria `kagglehub` per scaricare automaticamente il dataset nell'ambiente Google Colab. Successivamente, leggeremo il file CSV tramite Pandas e controlleremo le dimensioni e la struttura iniziale dei dati.


In [ ]:
import kagglehub
from pathlib import Path

# Download automatico del dataset
dataset_path = kagglehub.dataset_download(
    "saichaitanyareddyai/spotify-tracks-dataset-audio-features"
)

# Ricerca del file CSV
csv_files = list(Path(dataset_path).rglob("*.csv"))

if not csv_files:
    raise FileNotFoundError("Nessun file CSV trovato nel dataset.")

DATA_PATH = csv_files[0]

# Caricamento del dataset
df = pd.read_csv(DATA_PATH)

print("Dataset caricato correttamente!")
print("File:", DATA_PATH.name)
print("Dimensioni:", df.shape)

display(df.head())

# Verifica delle colonne e dei tipi di dato
print("Colonne del dataset:")
print(df.columns.tolist())

print("\nTipi di dato:")
display(df.dtypes)



# 3. Panoramica e qualità dei dati

*TODO: colonne, tipi, valori mancanti, duplicati e distribuzione delle label.*

---

## 3.1 Dati identificativi

### Track_id

È il codice identificativo univoco assegnato da Spotify alla specifica traccia presente nel catalogo. È una stringa alfanumerica utilizzata per identificare il brano e recuperare le relative informazioni attraverso i sistemi Spotify. Non rappresenta una caratteristica musicale.

### Artists

Indica il nome dell'artista o degli artisti che interpretano il brano. Nel caso di collaborazioni, possono essere presenti più nomi associati alla stessa traccia.

### Album_name

Indica il nome dell'album all'interno del quale è pubblicato il brano. Può riferirsi a un album, un EP, un singolo o un'altra tipologia di pubblicazione.

### Track_name

Indica il titolo ufficiale del brano così come registrato nel catalogo Spotify.

---

## 3.2 Informazioni generali

### Popularity

Indica il livello di popolarità del brano, espresso attraverso un punteggio intero compreso tra 0 e 100. Viene determinato da un algoritmo proprietario di Spotify, basato principalmente sul numero complessivo di riproduzioni (stream) e sulla loro recenza. I brani ascoltati frequentemente nel periodo più recente tendono ad avere un punteggio maggiore rispetto a quelli che hanno ricevuto molti ascolti in passato. Il valore non viene necessariamente aggiornato in tempo reale e può presentare alcuni giorni di ritardo. Spotify non rende pubblica la formula matematica utilizzata.

### Duration_ms

Indica la durata complessiva del brano, espressa in millisecondi. È un dato numerico relativo alla lunghezza della registrazione presente nel catalogo Spotify.

### Explicit

Indica la presenza di contenuti espliciti nel brano, attraverso un valore booleano: `True` se il brano è contrassegnato come contenente contenuti espliciti, `False` in caso contrario. La classificazione è un'informazione associata alla traccia da Spotify e non una misura numerica ricavata dalle altre caratteristiche audio. È da considerare che, secondo la documentazione Spotify, False può indicare anche una condizione non conosciuta.

### Track_genre

Identifica il genere musicale associato al brano, attraverso un'etichetta testuale. È una variabile categorica che permette di raggruppare le tracce in base al genere di appartenenza. Nel dataset rappresenta un'etichetta associata al brano e non un punteggio ricavato direttamente dalle caratteristiche audio. Non è documentata una formula universale di Spotify per determinare questa classificazione.

---

## 3.3 Caratteristiche audio

### Danceability

Indica quanto un brano sia adatto al ballo, con un valore compreso tra 0 e 1. Viene stimata considerando una combinazione di elementi musicali, tra cui il tempo (velocità), la stabilità ritmica, la forza dei battiti (beat strength) e la regolarità complessiva del ritmo. Un valore vicino a 1 indica una maggiore predisposizione al ballo, mentre un valore vicino a 0 indica una minore predisposizione.

### Energy

Misura il livello di intensità ed energia percepita del brano, con un valore compreso tra 0 e 1. È una caratteristica percettiva che tiene conto di diversi elementi sonori, tra cui l'ampiezza della dinamica (dynamic range), il volume percepito, il timbro, la frequenza degli attacchi sonori e l'entropia generale del segnale. Brani caratterizzati da suoni intensi, veloci e incisivi tendono ad avere valori elevati, mentre composizioni più delicate e meno intense tendono ad avere valori inferiori.

### Loudness

Rappresenta il livello di intensità sonora complessiva del brano, espresso in decibel (dB). Spotify fornisce una misura mediata sull'intera durata della traccia, utile per confrontare il livello sonoro relativo tra diversi brani. I valori si collocano generalmente tra -60 e 0 dB, ma possono assumere anche valori positivi. Valori numericamente maggiori indicano un livello sonoro più elevato rispetto al riferimento della misurazione.

### Speechiness

Misura la presenza di contenuti vocali parlati all'interno del brano, con un valore compreso tra 0 e 1. Valori inferiori a 0,33 indicano generalmente contenuti prevalentemente musicali; valori compresi tra 0,33 e 0,66 possono indicare una combinazione bilaciata di musica e parlato, come nel rap; valori superiori a 0,66 identificano brani caratterizzati prevalentemente da contenuti parlati.

### Acousticness

Indica il grado di affidabilità con cui Spotify identifica un brano come acustico, attraverso un valore compreso tra 0 e 1. Non rappresenta necessariamente una probabilità statistica in senso stretto, ma un indice di confidenza: valori vicini a 1 indicano una maggiore sicurezza nell'identificazione del brano come acustico, mentre valori vicini a 0 indicano caratteristiche meno riconducibili a una registrazione acustica.

### Instrumentalness

Stima la probabilità che il brano non contenga parti vocali, attraverso un valore compreso tra 0 e 1. Valori vicini a 1 indicano una maggiore probabilità che si tratti di un brano strumentale. Spotify considera strumentali anche alcune vocalizzazioni prive di parole, mentre riconosce il rap e il parlato come contenuti vocali. Valori superiori a 0,5 sono generalmente indicativi di brani strumentali, con una maggiore affidabilità man mano che il valore si avvicina a 1.

### Liveness

Indica la probabilità che il brano sia stato registrato durante un'esibizione dal vivo, con un valore compreso tra 0 e 1. La stima si basa sul rilevamento della presenza di un pubblico nella registrazione. Valori più elevati indicano una maggiore probabilità di una registrazione dal vivo; valori superiori a 0,8 costituiscono un forte indicatore di questa condizione.

### Valence

Rappresenta la positività emotiva trasmessa musicalmente dal brano, attraverso un valore compreso tra 0 e 1. Valori elevati identificano sonorità generalmente più positive, allegre, euforiche o vivaci, mentre valori bassi indicano sonorità percepite come più negative, malinconiche, tristi o cupe. Si riferisce alla positività musicale percepita e non necessariamente al significato del testo.

### Tempo

Indica la velocità complessiva del brano, espressa in battiti per minuto (BPM, Beats Per Minute). Viene stimata attraverso l'analisi del ritmo e della durata media dei battiti. Valori elevati indicano generalmente un ritmo più veloce, mentre valori bassi indicano un ritmo più lento.

---

## 3.4 Caratteristiche musicali

### Key

Indica la tonalità musicale predominante del brano, rappresentata da un valore intero compreso tra 0 e 11, secondo il sistema di notazione Pitch Class. Ogni valore corrisponde a una delle dodici note della scala cromatica: `0 = Do (C), 1 = Do♯/Re♭, 2 = Re (D), 3 = Re♯/Mi♭, 4 = Mi (E), 5 = Fa (F), 6 = Fa♯/Sol♭, 7 = Sol (G), 8 = Sol♯/La♭, 9 = La (A), 10 = La♯/Si♭, 11 = Si (B)`. Spotify stima la tonalità attraverso l'analisi del contenuto sonoro e delle altezze delle note. Se non viene identificata una tonalità, la documentazione prevede anche il valore -1.

### Mode

Indica la modalità della tonalità musicale, distinguendo tra modo maggiore e modo minore. È rappresentata da un valore binario: 1 corrisponde al modo maggiore, mentre 0 corrisponde al modo minore. Questa informazione riguarda il tipo di scala musicale da cui deriva il contenuto melodico del brano.

### Time_signature

Indica la divisione metrica del brano, ovvero il numero di battiti presenti all'interno di ciascuna battuta musicale. È espressa attraverso un valore intero stimato da Spotify. Ad esempio, il valore 3 corrisponde a un tempo di 3/4, il valore 4 a un tempo di 4/4 e il valore 5 a un tempo di 5/4. La documentazione Spotify prevede valori compresi tra 3 e 7, che rappresentano il numeratore della divisione metrica.



# 4. Grafici da utilizzare

I grafici sono suddivisi in due categorie:

- **PRINCIPALE:** grafico da realizzare perché contribuisce direttamente al raggiungimento dell'obiettivo della ricerca.
- **FACOLTATIVO:** grafico utile come approfondimento, ma non indispensabile.

## Conoscere e analizzare il dataset

### 1. Numero di brani per genere — PRINCIPALE

- **Tipo di grafico:** Bar Chart orizzontale.
- **Dati da utilizzare:** `track_genre`, numero di brani per ciascun genere.

- **Obiettivo:**
  - Capire quanti brani sono presenti per ogni genere.
  - Individuare i generi con pochi o molti brani.
  - Verificare se la distribuzione dei generi è equilibrata.

- **Motivo per realizzare il grafico:**
  - Permette di conoscere la composizione del dataset prima di iniziare l'analisi.
  - Aiuta a individuare eventuali squilibri nella distribuzione dei generi, che potrebbero influenzare le analisi successive.

- **Perché utilizzare questo tipo di grafico:**
  - Il Bar Chart è adatto a confrontare quantità appartenenti a categorie differenti.
  - L'orientamento orizzontale permette di visualizzare facilmente i nomi dei generi musicali, anche quando sono lunghi.
  - Ordinando le barre dalla più lunga alla più corta è immediato individuare i generi più e meno rappresentati.

### 2. Distribuzione delle caratteristiche musicali — PRINCIPALE

- **Tipo di grafico:** Istogrammi (possibilmente organizzati in Small Multiples).

- **Dati da utilizzare:**
  - `danceability`
  - `energy`
  - `valence`
  - `acousticness`
  - `instrumentalness`
  - `speechiness`
  - `tempo`
  - `loudness`

- **Obiettivo:**
  - Osservare come sono distribuite le caratteristiche musicali.
  - Capire quali valori sono più frequenti.
  - Individuare eventuali distribuzioni asimmetriche e valori estremi.

- **Motivo per realizzare il grafico:**
  - Permette di comprendere la distribuzione delle variabili che utilizzeremo per il clustering.
  - Aiuta a individuare caratteristiche con distribuzioni particolari o valori anomali, che potrebbero influenzare l'analisi.

- **Perché utilizzare questo tipo di grafico:**
  - L'istogramma permette di visualizzare la distribuzione di una variabile numerica.
  - Consente di capire quali intervalli di valori sono più frequenti e se la distribuzione è simmetrica oppure asimmetrica.
  - Utilizzando Small Multiples possiamo confrontare le distribuzioni delle otto caratteristiche attraverso grafici separati, senza sovraccaricare un'unica visualizzazione.

### 3. Correlazione tra le caratteristiche musicali — PRINCIPALE

- **Tipo di grafico:** Correlation Heatmap.
- **Dati da utilizzare:** le stesse otto caratteristiche musicali del grafico 2.

- **Obiettivo:**
  - Individuare le caratteristiche che tendono a variare insieme.
  - Individuare eventuali correlazioni positive o negative.
  - Capire se alcune variabili descrivono aspetti musicali simili.

- **Motivo per realizzare il grafico:**
  - Permette di comprendere le relazioni tra le variabili utilizzate nel clustering.
  - Aiuta a individuare eventuali correlazioni molto forti tra caratteristiche musicali.

- **Perché utilizzare questo tipo di grafico:**
  - La Heatmap utilizza colori differenti per rappresentare l'intensità e la direzione delle correlazioni.
  - Permette di confrontare rapidamente tutte le coppie di caratteristiche musicali.
  - È particolarmente utile quando si hanno diverse variabili, perché una tabella numerica sarebbe meno immediata da interpretare.

### 4. Differenze delle caratteristiche musicali tra generi — FACOLTATIVO

- **Tipo di grafico:** Box Plot.
- **Dati da utilizzare:**
  - `track_genre`
  - Un sottoinsieme delle otto caratteristiche musicali.

- **Obiettivo:**
  - Confrontare le caratteristiche musicali tra generi.
  - Osservare mediane, dispersione e valori estremi.
  - Capire se alcuni generi presentano caratteristiche simili o differenti.

- **Motivo per realizzare il grafico:**
  - Permette di effettuare un primo confronto tra le caratteristiche acustiche dei generi originali.
  - Aiuta a capire se esistono differenze misurabili tra le categorie prima di applicare il clustering.
  - È un approfondimento utile, ma alcune informazioni verranno riprese nel confronto finale tra cluster e generi.

- **Perché utilizzare questo tipo di grafico:**
  - Il Box Plot riassume la distribuzione di una variabile numerica attraverso mediana, quartili e valori anomali.
  - Permette di confrontare contemporaneamente la distribuzione di una caratteristica tra diversi generi.
  - È utile per individuare differenze e sovrapposizioni tra le categorie.
  
---

## Clustering: trovare gruppi senza conoscere il genere

### 1. Elbow Method — PRINCIPALE

- **Tipo di grafico:** Line Chart.

- **Dati da utilizzare:**
  - Le otto caratteristiche musicali standardizzate.
  - Numero di cluster `K`.
  - Inertia / WCSS calcolata da K-Means (somma dei quadrati delle distanze all'interno dei cluster).

- **Obiettivo:**
  - Provare diversi valori di `K`.
  - Osservare come diminuisce l'inerzia aumentando il numero di cluster.
  - Individuare il possibile punto di equilibrio per scegliere `K`.

- **Motivo per realizzare il grafico:**
  - Permette di individuare un possibile numero di cluster appropriato per raggruppare i brani.
  - Evita di scegliere il numero di gruppi in modo arbitrario.

- **Perché utilizzare questo tipo di grafico:**
  - Il Line Chart è adatto a mostrare come cambia una misura al variare di un parametro.
  - Permette di osservare l'andamento della WCSS quando aumenta il numero di cluster `K`.
  - Rende visibile il possibile punto di gomito, nel quale il miglioramento ottenuto aggiungendo altri cluster diventa meno significativo.

### 2. Silhouette Score — PRINCIPALE

- **Tipo di grafico:** Line Chart.

- **Dati da utilizzare:**
  - Le otto caratteristiche musicali standardizzate.
  - Diversi valori di `K`.
  - Silhouette Score calcolato per ogni valore di `K`.

- **Obiettivo:**
  - Valutare quanto i cluster sono compatti e separati tra loro.
  - Confrontare la qualità delle diverse configurazioni.
  - Affiancare questa valutazione all'Elbow Method nella scelta di `K`.

- **Motivo per realizzare il grafico:**
  - Permette di valutare se i gruppi individuati da K-Means sono effettivamente ben separati.
  - Completa l'Elbow Method, fornendo un'altra misura per scegliere il numero di cluster.

- **Perché utilizzare questo tipo di grafico:**
  - Permette di confrontare il Silhouette Score ottenuto con diversi valori di `K`.
  - La linea rende immediatamente visibili eventuali aumenti e diminuzioni della qualità del clustering.
  - Facilita l'individuazione dei valori di `K` che producono una migliore separazione tra i gruppi.

### 3. Visualizzazione dei cluster tramite PCA — PRINCIPALE

- **Tipo di grafico:** Scatter Plot.

- **Dati da utilizzare:**
  - Le otto caratteristiche musicali standardizzate.
  - Etichetta del cluster assegnata da K-Means.
  - Prime due componenti principali ottenute con PCA.

- **Obiettivo:**
  - Rappresentare visivamente i gruppi individuati.
  - Osservare se i cluster appaiono separati o sovrapposti.
  - Individuare eventuali gruppi molto vicini tra loro.

- **Motivo per realizzare il grafico:**
  - Permette di visualizzare in due dimensioni i gruppi individuati utilizzando otto caratteristiche musicali.
  - Aiuta a interpretare visivamente la distribuzione dei cluster.

- **Perché utilizzare questo tipo di grafico:**
  - Lo Scatter Plot permette di rappresentare osservazioni individuali attraverso punti in uno spazio bidimensionale.
  - Ogni punto rappresenta un brano e il colore identifica il cluster di appartenenza.
  - Consente di osservare la vicinanza tra i brani, la separazione tra i gruppi e le eventuali sovrapposizioni.
  - La PCA permette di rappresentare in due dimensioni informazioni originariamente distribuite su otto caratteristiche.

**Nota:** la PCA viene utilizzata per ridurre le dimensioni e visualizzare i dati in due dimensioni. K-Means viene applicato alle otto caratteristiche standardizzate originali, non alle sole due componenti PCA.

### 4. Profilo musicale dei cluster — PRINCIPALE

- **Tipo di grafico:** Heatmap.

- **Dati da utilizzare:**
  - Etichetta `cluster` ottenuta da K-Means.
  - Le otto caratteristiche musicali.
  - Media di ciascuna caratteristica per cluster.

- **Obiettivo:**
  - Analizzare le caratteristiche medie di ogni cluster.
  - Capire quali variabili caratterizzano maggiormente ciascun gruppo.
  - Attribuire un'interpretazione musicale ai cluster.

- **Motivo per realizzare il grafico:**
  - Permette di comprendere quali caratteristiche acustiche contraddistinguono i gruppi individuati.
  - Aiuta a interpretare musicalmente i cluster, andando oltre la semplice visualizzazione dei punti.

- **Perché utilizzare questo tipo di grafico:**
  - La Heatmap permette di confrontare contemporaneamente più caratteristiche per ciascun cluster.
  - L'utilizzo dei colori facilita l'individuazione delle caratteristiche con valori medi più alti o più bassi.
  - È particolarmente adatta a una matrice con i cluster sulle righe e le caratteristiche musicali sulle colonne.

### 5. Confronto tra cluster e generi musicali — PRINCIPALE

- **Tipo di grafico:** Heatmap.

- **Dati da utilizzare:**
  - `cluster` assegnato da K-Means.
  - `track_genre` originale.
  - Conteggi e percentuali per ogni combinazione cluster-genere.

- **Obiettivo:**
  - Confrontare i cluster individuati autonomamente con i generi originali.
  - Osservare quali generi sono presenti in ciascun cluster.
  - Individuare generi distribuiti in più cluster.
  - Individuare cluster che contengono generi differenti.
  - Verificare quanto i gruppi acustici corrispondono alle categorie di genere.

- **Motivo per realizzare il grafico:**
  - È il grafico più importante per rispondere alla domanda di ricerca.
  - Permette di verificare se i gruppi individuati attraverso le caratteristiche musicali corrispondono ai generi assegnati nel dataset oppure se generi diversi vengono raggruppati insieme.
  - Aiuta a individuare eventuali sovrapposizioni tra categorie musicali.

- **Perché utilizzare questo tipo di grafico:**
  - La Heatmap permette di rappresentare una relazione tra due variabili categoriali: cluster e genere musicale.
  - I colori evidenziano quali combinazioni cluster-genere sono più frequenti.
  - Utilizzando percentuali per riga è possibile confrontare la composizione dei cluster anche quando hanno dimensioni differenti.
  - Consente di individuare immediatamente se un cluster contiene prevalentemente un genere oppure se raccoglie brani appartenenti a generi differenti.

**Nota:** utilizzare percentuali per riga, in modo che ogni cluster mostri la propria composizione interna. Considerando la presenza di 114 generi, potrebbe essere necessario selezionare o aggregare alcune categorie secondo un criterio dichiarato, per mantenere leggibile la visualizzazione.

---

## Classificazione: verificare quanto le caratteristiche predicono il genere — FACOLTATIVA

Questa fase rappresenta un approfondimento rispetto all'obiettivo principale. A differenza del clustering, utilizza i generi originali come etichette da prevedere.

### 1. Decision Tree — FACOLTATIVO

- **Tipo di grafico:** Decision Tree.

- **Dati da utilizzare:**
  - Input: le otto caratteristiche musicali.
  - Target: `track_genre`.

- **Obiettivo:**
  - Costruire un modello che provi a prevedere il genere.
  - Osservare quali caratteristiche vengono utilizzate nelle decisioni.
  - Comprendere le regole di classificazione.

- **Motivo per realizzare il grafico:**
  - Permette di verificare se le caratteristiche musicali possono essere utilizzate per riconoscere e prevedere i generi.
  - Offre un confronto aggiuntivo rispetto ai risultati ottenuti con il clustering.

- **Perché utilizzare questo tipo di grafico:**
  - L'albero decisionale rappresenta le regole utilizzate dal modello attraverso nodi e rami.
  - Permette di seguire il percorso delle decisioni e comprendere quali caratteristiche vengono utilizzate per classificare un brano.
  - È una visualizzazione interpretabile, utile per comprendere il funzionamento del modello.

### 2. Feature Importance — FACOLTATIVO

- **Tipo di grafico:** Bar Chart orizzontale.

- **Dati da utilizzare:**
  - Modello Random Forest.
  - Le otto caratteristiche musicali.
  - Valori di feature importance.

- **Obiettivo:**
  - Individuare le caratteristiche più utilizzate dal modello.
  - Ordinare le variabili in base alla loro importanza.
  - Capire quali proprietà musicali contribuiscono maggiormente alla previsione del genere.

- **Motivo per realizzare il grafico:**
  - Permette di capire quali caratteristiche musicali risultano più informative per il modello nella classificazione dei generi.
  - Aggiunge un'interpretazione dei risultati della classificazione.

- **Perché utilizzare questo tipo di grafico:**
  - Il grafico a barre permette di confrontare e ordinare l'importanza delle diverse caratteristiche.
  - L'orientamento orizzontale rende leggibili i nomi delle variabili.
  - Consente di individuare rapidamente quali caratteristiche risultano più informative per il modello.

### 3. Confronto Decision Tree vs Random Forest — FACOLTATIVO

- **Tipo di grafico:** Bar Chart comparativo.

- **Dati da utilizzare:**
  - Decision Tree.
  - Random Forest.
  - Accuracy calcolata sullo stesso test set.

- **Obiettivo:**
  - Confrontare le prestazioni dei due algoritmi.
  - Verificare quale riesce a prevedere meglio il genere.
  - Capire se un modello più complesso migliora i risultati.

- **Motivo per realizzare il grafico:**
  - Permette di confrontare due approcci di classificazione.
  - Aiuta a capire se Random Forest offre risultati migliori rispetto a un singolo Decision Tree.

- **Perché utilizzare questo tipo di grafico:**
  - Le barre permettono di confrontare direttamente le prestazioni dei due modelli.
  - Utilizzando la stessa scala è possibile osservare facilmente quale modello ottiene il risultato migliore.
  - È una soluzione semplice e immediata per confrontare le metriche di valutazione.

### 4. Confusion Matrix — FACOLTATIVO

- **Tipo di grafico:** Confusion Matrix / Heatmap.

- **Dati da utilizzare:**
  - Generi reali nel test set.
  - Generi previsti dal modello.
  - Risultati del Decision Tree o Random Forest.

- **Obiettivo:**
  - Osservare quali generi vengono classificati correttamente.
  - Individuare quali generi vengono confusi tra loro.
  - Analizzare eventuali somiglianze tra categorie.

- **Motivo per realizzare il grafico:**
  - Permette di approfondire gli errori commessi dal modello.
  - Aiuta a individuare quali generi sono più difficili da distinguere attraverso le caratteristiche musicali.
  - Completa l'analisi delle prestazioni dei modelli.

- **Perché utilizzare questo tipo di grafico:**
  - La Heatmap permette di rappresentare il confronto tra classi reali e classi previste.
  - I colori rendono immediatamente visibili le classificazioni corrette e gli errori.
  - Consente di individuare quali generi vengono confusi più frequentemente dal modello.

## 5. Preprocessing e normalizzazione

TODO: selezionare le feature e motivare le trasformazioni in base all'algoritmo.


## 6. Applicazione dell'algoritmo centrale

TODO: inserire spiegazione, parametri, applicazione e valutazione del metodo scelto dopo averlo verificato sulle slide.


## 7. Interpretazione dei risultati

TODO: visualizzare i risultati, individuare le feature importanti e confrontare con la label quando pertinente.


## 8. Conclusioni e limiti

TODO: rispondere alle domande di ricerca, discutere limiti e possibili sviluppi.
